# SENTINEL ANPR - Model 3 License Plate Detector Fine-Tuning (Colab T4 GPU)

This notebook trains the SENTINEL license plate detector (`plate_single_line` and `plate_two_line`) on Gujarat Police live CCTV frames using Google Colab's free T4 GPU.

### Pipeline Steps:
1. Environment check & GPU verification (`nvidia-smi`)
2. Ultralytics YOLOv8 installation
3. Dataset unpacking / configuration
4. Pretrained baseline evaluation (mAP50, Precision, Recall)
5. 50-epoch GPU fine-tuning (transfer learning)
6. Post-training validation & weights export

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install -q ultralytics opencv-python pyyaml

In [ ]:
# 3. Clone repo or upload dataset zip
# If repo is public/accessible:
!git clone https://github.com/mitishapatidar/sentinel-cctv.git
%cd sentinel-cctv

# If running standalone with uploaded sentinel_dataset.zip:
# !unzip -q /content/sentinel_dataset.zip -d /content/dataset

In [ ]:
# 4. Baseline Evaluation (Pretrained Model before fine-tuning)
from ultralytics import YOLO

baseline_model = YOLO('yolov8n.pt')
print('Evaluating baseline on validation set...')
baseline_metrics = baseline_model.val(data='ai_pipeline/dataset/final/data.yaml', split='val', imgsz=640)

print(f'Baseline mAP50: {baseline_metrics.box.map50:.4f}')
print(f'Baseline Precision: {baseline_metrics.box.mp:.4f}')
print(f'Baseline Recall: {baseline_metrics.box.mr:.4f}')

In [ ]:
# 5. Full 50-Epoch GPU Fine-Tuning (Transfer Learning)
model = YOLO('yolov8n.pt')

results = model.train(
    data='ai_pipeline/dataset/final/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=15,
    save=True,
    project='runs/detect',
    name='sentinel_plate_detector',
    exist_ok=True
)

In [ ]:
# 6. Post-Training Evaluation on Validation Set
best_model = YOLO('runs/detect/sentinel_plate_detector/weights/best.pt')
eval_results = best_model.val(data='ai_pipeline/dataset/final/data.yaml', split='val', imgsz=640)

print('=' * 60)
print('POST-TRAINING FINE-TUNED METRICS (VAL SET)')
print('=' * 60)
print(f'mAP@50:    {eval_results.box.map50:.4f}')
print(f'mAP@50-95: {eval_results.box.map:.4f}')
print(f'Precision: {eval_results.box.mp:.4f}')
print(f'Recall:    {eval_results.box.mr:.4f}')
print('=' * 60)

In [ ]:
# 7. Download best.pt checkpoint to local machine
from google.colab import files
files.download('runs/detect/sentinel_plate_detector/weights/best.pt')